# Edge-Native Hybrid ChakraModel Evaluation
This notebook evaluates the fully combined YOLO + ChakraTransformer architecture across all datasets using cloud GPUs.

In [ ]:
!pip install -q ultralytics timm opencv-python-headless torchvision

import sys
import os
import glob
import shutil
from pathlib import Path

print("Looking for attached code dataset...")
explicit_dirs = [
    '/kaggle/input/om-finalkaggle-upload',
    '/kaggle/input/chakramodel-kaggle-code',
    '/kaggle/input/datasets/gokulrocky/om-finalkaggle-upload',
    '/kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code'
]
base_dirs = explicit_dirs + glob.glob('/kaggle/input/*/*') + glob.glob('/kaggle/input/*')
src_dir = None
for d in base_dirs:
    if os.path.exists(os.path.join(d, 'src', 'evaluate_all.py')):
        src_dir = d
        break

if src_dir:
    print(f"Found codebase at: {src_dir}")
    print("Copying to working directory to allow execution...")
    if os.path.exists('/kaggle/working/src'): shutil.rmtree('/kaggle/working/src')
    if os.path.exists('/kaggle/working/weights'): shutil.rmtree('/kaggle/working/weights')
    shutil.copytree(os.path.join(src_dir, 'src'), '/kaggle/working/src')
    shutil.copytree(os.path.join(src_dir, 'weights'), '/kaggle/working/weights')
    sys.path.insert(0, '/kaggle/working/src')
else:
    print("ERROR: Could not find 'src/evaluate_all.py'.")


## Setup Datasets


In [ ]:
os.makedirs('/kaggle/working/data', exist_ok=True)

def auto_symlink(dataset_name, internal_name):
    # Check explicit paths first
    paths = glob.glob(f'/kaggle/input/**/*{dataset_name}*/images', recursive=True) + \
            glob.glob(f'/kaggle/input/**/*{dataset_name}*/Original', recursive=True) + \
            glob.glob(f'/kaggle/input/chakramodel-evaluation-datasets/**/*{dataset_name}*/images', recursive=True) + \
            glob.glob(f'/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/**/*{dataset_name}*/images', recursive=True)
    
    if not paths:
        paths = glob.glob(f'/kaggle/input/**/*{internal_name}*/images', recursive=True) + \
                glob.glob(f'/kaggle/input/chakramodel-evaluation-datasets/**/*{internal_name}*/images', recursive=True) + \
                glob.glob(f'/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets/**/*{internal_name}*/images', recursive=True)
        
    if paths:
        root_ds = Path(paths[0]).parent
        target = f'/kaggle/working/data/{internal_name}'
        if not os.path.exists(target):
            os.symlink(str(root_ds), target)
            print(f'✅ Linked {internal_name} from {root_ds}')
    else:
        print(f'⚠️ WARNING: {internal_name} not found.')

auto_symlink('kvasir', 'kvasir-seg')
auto_symlink('clinic', 'cvc-clinicdb')
auto_symlink('etis', 'etis-larib')
auto_symlink('colon', 'cvc-colondb')
auto_symlink('300', 'cvc-300')


## Run Final Evaluation
This will run `evaluate_all.py` which loads both the YOLO detection model and the ChakraTransformer segmentation model, simulating the Edge-Native Hybrid architecture pipeline!

In [ ]:
!python /kaggle/working/src/evaluate_all.py